# Runtime Environment Setup

In [106]:
!git clone https://github.com/ethan-yountz/560-FaceRecognition

Cloning into '560-FaceRecognition'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 77 (delta 35), reused 66 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 88.62 KiB | 5.54 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [107]:
import os
os.chdir("560-FaceRecognition/project-fr")

In [108]:
!pwd
!ls

/content/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr
evaluate.py		  models     run_baseline_benchmark.py
make_validation_split.py  README.md  train_example.py


In [109]:
from pathlib import Path
from google.colab import drive
import subprocess

drive.mount("/content/drive", force_remount=True)

drive_root = Path("/content/drive/MyDrive")
comp560_dir = drive_root / "Comp560"
candidate_paths = [
    comp560_dir / "datasets.tar.gz",
    drive_root / "datasets.tar.gz",
]
datasets_tar = next((path for path in candidate_paths if path.exists()), None)

if datasets_tar is None:
    if comp560_dir.exists():
        visible_files = sorted(path.name for path in comp560_dir.iterdir())[:20]
        raise FileNotFoundError(
            f"Could not find datasets.tar.gz. Checked: {candidate_paths}. Files in {comp560_dir}: {visible_files}"
        )
    raise FileNotFoundError(
        f"Could not find datasets.tar.gz. Checked: {candidate_paths}. Folder not found: {comp560_dir}"
    )

subprocess.run(["tar", "-xf", str(datasets_tar)], check=True)
print(f"Extracted {datasets_tar} into {Path.cwd()}")

Mounted at /content/drive
Extracted /content/drive/MyDrive/Comp560/datasets.tar.gz into /content/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr/560-FaceRecognition/project-fr


In [110]:
from pathlib import Path
import subprocess

dataset_dir = Path("datasets/dataset_a")
if not dataset_dir.exists():
    available_datasets = sorted(path.name for path in Path("datasets").iterdir() if path.is_dir())
    raise FileNotFoundError(
        f"Missing dataset_a after extraction. Available dataset directories: {available_datasets}"
    )

images_tar = dataset_dir / "images.tar.gz"
if not images_tar.exists():
    raise FileNotFoundError(f"Missing image archive: {images_tar}")

subprocess.run(["tar", "-xf", images_tar.name], cwd=images_tar.parent, check=True)
print(f"Extracted {images_tar}")

Extracted datasets/dataset_a/images.tar.gz


In [111]:
from pathlib import Path
import time

save_target = Path("/content/drive/MyDrive/Comp560/saves")
save_target.mkdir(parents=True, exist_ok=True)
save_link = Path("saves")

if save_link.is_symlink():
    if save_link.resolve() != save_target.resolve():
        save_link.unlink()
        save_link.symlink_to(save_target, target_is_directory=True)
elif save_link.exists():
    raise FileExistsError(f"{save_link} already exists and is not a symlink.")
else:
    save_link.symlink_to(save_target, target_is_directory=True)

with open(save_link / "logs.txt", "a", encoding="utf-8") as handle:
    handle.write(time.strftime("%Y-%m-%d %H:%M:%S\n"))

print(f"Using save directory: {save_link.resolve()}")

Using save directory: /content/drive/MyDrive/Comp560/saves


In [112]:
from pathlib import Path
import subprocess

split_dir = Path("datasets/dataset_a/splits/val_15_seed42")
required_split_files = [
    split_dir / "train_metadata.parquet",
    split_dir / "val_metadata.parquet",
    split_dir / "val_pairs.parquet",
]
missing_split_files = [str(path) for path in required_split_files if not path.exists()]

if missing_split_files:
    print(f"Creating held-out split because these files are missing: {missing_split_files}")
    subprocess.run([
        "python",
        "make_validation_split.py",
        "--dataset_root",
        "datasets/dataset_a",
        "--val_fraction",
        "0.15",
        "--seed",
        "42",
    ], check=True)
else:
    print(f"Using existing held-out split: {split_dir}")

Creating held-out split because these files are missing: ['datasets/dataset_a/splits/val_15_seed42/train_metadata.parquet', 'datasets/dataset_a/splits/val_15_seed42/val_metadata.parquet', 'datasets/dataset_a/splits/val_15_seed42/val_pairs.parquet']


## Command Arguments

In [113]:
training_args = """
--data_root datasets/dataset_a
--save_dir saves/
--train_metadata splits/val_15_seed42/train_metadata.parquet
--val_metadata splits/val_15_seed42/val_metadata.parquet
--val_pairs splits/val_15_seed42/val_pairs.parquet
--backbone mobilefacenet
--embedding_dim 128
--epochs 1
""".replace("\n", " ")

prediction_args = """
--predict
--checkpoint saves/best_model.pth
--dataset_root datasets/dataset_a
--output predictions/dataset_a.csv
""".replace("\n", " ")

shared_args = """
--batch_size 256
--num_workers 8
--device cuda
""".replace("\n", " ")

evaluation_args = """
--student_id 730702719
--prediction predictions/dataset_a.csv
--acknowledge_benchmark_labels
--datasets dataset_a
""".replace("\n", " ")

# Train Model

In [114]:
cmd = f"python train_example.py {training_args} {shared_args}"
print(cmd)

python train_example.py  --data_root datasets/dataset_a --save_dir saves/ --train_metadata splits/val_15_seed42/train_metadata.parquet --val_metadata splits/val_15_seed42/val_metadata.parquet --val_pairs splits/val_15_seed42/val_pairs.parquet --backbone mobilefacenet --embedding_dim 128 --epochs 10   --batch_size 256 --num_workers 8 --device cuda 


In [ ]:
!{cmd}

Training set: 199323 images, 1571 classes using label 'component_id'
Epoch 1/10:   0% 0/778 [00:00<?, ?it/s]

# Evaluate Model
## Generate Predictions

In [ ]:
cmd = f"python train_example.py {prediction_args} {shared_args}"
print(cmd)

In [ ]:
!{cmd}

## Evaluate

In [ ]:
cmd = f"python evaluate.py {evaluation_args}"
print(cmd)

In [ ]:
!{cmd}